# SVGen: train the 1.5B model on Google Colab

Fine-tunes **Qwen2.5-Coder-1.5B** with LoRA on the SVGen icon dataset. That's 3× larger than the 0.5B model trained on the Mac.

**Before you start:** `Runtime → Change runtime type → GPU` (T4 is free; L4 or A100 with Colab Pro are much faster).

Run the cells top to bottom. Checkpoints go to your Google Drive. **If Colab disconnects, run all cells again: training continues where it stopped.**

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Connect Google Drive (checkpoints and the final model are saved here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/svgen/svgen-1.5b'

## 3. Get the code and install

In [ ]:
%cd /content
!test -d SVGen || git clone -q https://github.com/artamrj/SVGen.git
%cd /content/SVGen
!git pull -q
!pip install -q -U transformers peft

## 4. Build the dataset (~10 MB download, under a minute)

In [ ]:
!python -m svgen.download
!python -m svgen.data | tail -12

## 5. Train

| GPU | 3 epochs, roughly |
|---|---|
| T4 (free) | 4–6 hours, may need a reconnect |
| L4 | ~2 hours |
| A100 | ~1 hour |

Watch `loss` and `val_loss` go down. `eta_min` is the minutes left. On a free T4 you can set `EPOCHS = 2` to finish sooner.

In [ ]:
import torch
MODEL = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
EPOCHS = 3
big_gpu = torch.cuda.get_device_properties(0).total_memory > 30e9
BATCH, ACCUM = (8, 2) if big_gpu else (4, 4)   # effective batch 16 either way
!python -m svgen.train_cuda --model {MODEL} --epochs {EPOCHS} --batch-size {BATCH} --grad-accum {ACCUM} --out "{OUT}"

## 6. Try it

In [ ]:
import torch
from IPython.display import SVG, display
from transformers import AutoModelForCausalLM, AutoTokenizer
from svgen.prompting import messages
from svgen.svg import SvgError, parse_body, salvage_body, wrap_body

tok = AutoTokenizer.from_pretrained(f'{OUT}/merged')
model = AutoModelForCausalLM.from_pretrained(f'{OUT}/merged', dtype=torch.float16).cuda().eval()

def draw(prompt, n=4):
    text = tok.apply_chat_template(messages(prompt), add_generation_prompt=True, tokenize=False)
    ids = tok(text, return_tensors='pt').to('cuda')
    out = model.generate(**ids, max_new_tokens=700, do_sample=True, temperature=0.6, top_p=0.95, num_return_sequences=n)
    bodies = [salvage_body(tok.decode(o[ids.input_ids.shape[1]:], skip_special_tokens=True)) for o in out]
    return [wrap_body(b, size=96) for b in bodies if b]

for prompt in ['star', 'rocket', 'bicycle', 'umbrella', 'camera', 'cloud with rain']:
    print(prompt)
    display(*[SVG(s) for s in draw(prompt)])

## 7. Move the model to your Mac

1. In Google Drive, download the folder `svgen/svgen-1.5b/merged` (~3 GB).
2. On the Mac, in the SVGen folder:

```bash
make convert MERGED=~/Downloads/merged NAME=svgen-1.5b
make svg PROMPT="rocket" BASE=models/svgen-1.5b ADAPTER=none OUT=rocket.svg
make eval BASE=models/svgen-1.5b ADAPTER=none
```

`make convert` turns it into an 8-bit MLX model (~1.7 GB) that runs fast on Apple Silicon.